In [9]:
%pip install sentencepiece protobuf==3.20.3
%pip install --upgrade transformers tokenizers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import torch

In [ ]:
from transformers import AutoTokenizer, AutoModel, BertModel, BertTokenizer

# bert: bert-base-uncased,
# biobert: dmis-lab/biobert-base-cased-v1.1; use BertTokenizer and BertModel instead of AutoTokenizer and AutoModel
# chemberta v2: DeepChem/ChemBERTa-77M-MTR
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model.eval()

c:\Users\Diya Budlakoti\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Diya Budlakoti\.cache\huggingface\hub\models--dmis-lab--biobert-base-cased-v1.1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6416.07

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [5]:
def get_embedding(smiles):
    inputs = tokenizer(
        smiles,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # CLS token embedding (standard choice)
    embedding = outputs.last_hidden_state[:, 0, :]
    
    return embedding.squeeze(0)

In [6]:
def batch_embeddings(smiles_list, batch_size=32):
    all_embeddings = []
    
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        batch_emb = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(batch_emb)
    
    return torch.cat(all_embeddings, dim=0)

In [7]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Diya Budlakoti\\Desktop\\SIN_proj\\dataset\\final_processed_data.csv")
unique_smiles = list(set(df["SMILES1"]) | set(df["SMILES2"]))

unique_embeddings = batch_embeddings(unique_smiles)

# Create mapping
smiles_to_emb = dict(zip(unique_smiles, unique_embeddings))
emb_s1 = torch.stack([smiles_to_emb[s] for s in df["SMILES1"]])
emb_s2 = torch.stack([smiles_to_emb[s] for s in df["SMILES2"]])

In [8]:
combined_embeddings = []

for e1, e2 in zip(emb_s1, emb_s2):
    combined_embeddings.append(e1 + e2)

combined_embeddings = torch.stack(combined_embeddings)

In [9]:
embedding_cache = {}
def get_embedding_cached(smiles):
    if smiles in embedding_cache:
        return embedding_cache[smiles]
    
    emb = get_embedding(smiles)
    embedding_cache[smiles] = emb
    return emb

In [10]:
for s in df["SMILES1"]:
    get_embedding_cached(s)

for s in df["SMILES2"]:
    get_embedding_cached(s)
print(len(embedding_cache))

644


In [ ]:
torch.save(embedding_cache, "bert_smiles_embeddings.pt")